# 1. Download SPY 1-minute data

Run this notebook **once**. It downloads 1-minute SPY bars from Alpaca (free plan, history since 2016),
keeps the regular session (09:30-15:59 New York time) and saves the result to Google Drive.

The API keys are read from Colab Secrets (`ALPACA_KEY`, `ALPACA_SECRET`) and are never written in the code.

In [ ]:
!pip install alpaca-py --quiet

from datetime import datetime
import pandas as pd
from google.colab import drive, userdata
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame

DATA_FILE = "/content/drive/MyDrive/spy_1min.parquet"
FIRST_YEAR = 2016
LAST_DAY = datetime(2026, 10, 1)

client = StockHistoricalDataClient(userdata.get("ALPACA_KEY"), userdata.get("ALPACA_SECRET"))

In [ ]:
def download_year(year):
    """Download one year of 1-minute SPY bars, regular session only."""
    request = StockBarsRequest(
        symbol_or_symbols="SPY",
        timeframe=TimeFrame.Minute,
        start=datetime(year, 1, 1),
        end=min(datetime(year + 1, 1, 1), LAST_DAY),
    )
    bars = client.get_stock_bars(request).df
    bars = bars.reset_index(level="symbol", drop=True)
    bars.index = bars.index.tz_convert("America/New_York")
    return bars.between_time("09:30", "15:59")


pieces = []
for year in range(FIRST_YEAR, LAST_DAY.year + 1):
    piece = download_year(year)
    print(year, len(piece))
    pieces.append(piece)

spy = pd.concat(pieces)
print("Total rows:", len(spy))

In [ ]:
drive.mount("/content/drive")
spy.to_parquet(DATA_FILE)
print("Saved to", DATA_FILE)